# Pandas Complete Tutorial

This notebook explains Pandas from basics to advanced data manipulation in a clean, structured way.

You will learn with small example DataFrames first, then use `obesity_prediction.csv` for realistic practice.

## How to use this notebook

- Run cells from top to bottom.
- Read the markdown before each code cell.
- Use comments in code cells as revision notes.
- Change values and rerun cells to test your understanding.

## Topics covered

1. Setup and display options
2. Series and DataFrames
3. Reading and writing data
4. Inspecting data
5. Selecting rows and columns
6. Filtering and sorting
7. Creating, updating, and deleting columns
8. Missing values
9. Duplicates and cleaning
10. Data types
11. String operations
12. Date and time operations
13. GroupBy, aggregation, transform, and filter
14. Reshaping with melt, pivot, pivot table, stack, and unstack
15. Concat and all important joins
16. Indexes and MultiIndex
17. Rolling, expanding, cumulative calculations, ranking, and binning
18. Method chaining, plotting, performance tips, and practice tasks

## 1. Setup

Import Pandas and NumPy. NumPy helps with numeric operations and missing values.

In [ ]:
import numpy as np
import pandas as pd

# Display options make notebook outputs easier to read.
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 120)
pd.set_option("display.precision", 2)

print("Pandas version:", pd.__version__)

## 2. Series

A `Series` is a one-dimensional labeled array. Think of it as one column with an index.

In [ ]:
marks = pd.Series(
    data=[86, 78, 72, 91],
    index=["Aarav", "Aditi", "Vihaan", "Diya"],
    name="Marks"
)

marks

In [ ]:
# Basic Series operations.
print("Mean:", marks.mean())
print("Highest:", marks.max())
print("Students above 80:")
print(marks[marks > 80])

## 3. DataFrames

A `DataFrame` is a two-dimensional table with rows and columns.

In [ ]:
students = pd.DataFrame({
    "Student_ID": [1, 2, 3, 4, 5],
    "Name": ["Aarav", "Aditi", "Vihaan", "Diya", "Kabir"],
    "Age": [17, 18, 17, 19, 18],
    "Gender": ["Male", "Female", "Male", "Female", "Male"],
    "Marks": [86, 78, 72, 91, 67],
    "City": ["Bengaluru", "Mumbai", "Delhi", "Bengaluru", "Pune"]
})

students

In [ ]:
# Useful DataFrame attributes.
print("Shape:", students.shape)
print("Columns:", students.columns.tolist())
print("Index:", students.index.tolist())
print("Data types:")
print(students.dtypes)

## 4. Reading CSV Data

Use `pd.read_csv()` for CSV files. This project includes `obesity_prediction.csv`.

In [ ]:
obesity_data = pd.read_csv("obesity_prediction.csv")
obesity_data.head()

In [ ]:
# info() gives column names, non-null counts, and data types.
obesity_data.info()

In [ ]:
# describe() summarizes numeric columns.
obesity_data.describe()

In [ ]:
# include="object" summarizes text/categorical columns.
obesity_data.describe(include="object")

## 5. Writing Data

Pandas can export data to CSV, Excel, JSON, SQL, Parquet, and more.

In [ ]:
# Export examples are commented so they do not create files automatically.
# students.to_csv("students_export.csv", index=False)
# students.to_excel("students_export.xlsx", index=False)
# students.to_json("students_export.json", orient="records", indent=2)

# Use index=False when the row index is not meaningful data.

## 6. Inspecting Data

Before analysis, check shape, columns, missing values, unique values, and sample rows.

In [ ]:
print("Rows and columns:", obesity_data.shape)
print("Columns:")
print(obesity_data.columns.tolist())

In [ ]:
# Missing values per column.
obesity_data.isna().sum()

In [ ]:
# Unique values per column.
obesity_data.nunique()

In [ ]:
# Frequency counts for a category.
obesity_data["Obesity_Level"].value_counts()

## 7. Selecting Columns

Use square brackets to select one or more columns.

In [ ]:
# One column returns a Series.
students["Name"]

In [ ]:
# Multiple columns return a DataFrame.
students[["Name", "Age", "Marks"]]

In [ ]:
obesity_data[["Name", "Age", "Height_cm", "Weight_kg", "BMI"]].head()

## 8. Selecting Rows With `loc`, `iloc`, `at`, and `iat`

- `loc` selects by labels.
- `iloc` selects by integer positions.
- `at` gets or sets one value by label.
- `iat` gets or sets one value by integer position.

In [ ]:
# loc: row labels and column labels.
students.loc[0:2, ["Name", "Marks"]]

In [ ]:
# iloc: row positions and column positions.
students.iloc[0:3, 1:4]

In [ ]:
print("at example:", students.at[0, "Name"])
print("iat example:", students.iat[0, 1])

In [ ]:
obesity_data.loc[[0, 10, 20], ["Student_ID", "Name", "Age", "BMI", "Obesity_Level"]]

## 9. Filtering Rows

Filtering uses boolean conditions. Use `&` for AND, `|` for OR, and `~` for NOT.

In [ ]:
# Students with marks greater than or equal to 80.
students[students["Marks"] >= 80]

In [ ]:
# Multiple conditions require parentheses.
students[(students["Age"] >= 18) & (students["Marks"] >= 75)]

In [ ]:
# isin() checks whether values are in a list.
students[students["City"].isin(["Bengaluru", "Pune"])]

In [ ]:
# between() is useful for numeric ranges.
obesity_data[obesity_data["BMI"].between(25, 30)].head()

In [ ]:
# query() can make filters easier to read.
obesity_data.query("Age >= 18 and Weight_kg > 75 and Height_cm < 175").head()

## 10. Sorting

Use `sort_values()` for columns and `sort_index()` for index labels.

In [ ]:
students.sort_values("Marks", ascending=False)

In [ ]:
obesity_data.sort_values(["Obesity_Level", "BMI"], ascending=[True, False]).head(10)

## 11. Creating, Updating, and Deleting Columns

Most feature engineering starts by creating new columns from existing columns.

In [ ]:
student_features = students.copy()

# Create a boolean column.
student_features["Passed"] = student_features["Marks"] >= 70

# Create a grade column using multiple conditions.
conditions = [
    student_features["Marks"] >= 85,
    student_features["Marks"] >= 70,
    student_features["Marks"] < 70,
]
choices = ["A", "B", "Needs Improvement"]
student_features["Grade"] = np.select(conditions, choices, default="Unknown")

student_features

In [ ]:
health = obesity_data.copy()

# Recalculate BMI from height and weight.
health["BMI_Recalculated"] = health["Weight_kg"] / (health["Height_cm"] / 100) ** 2
health[["Name", "BMI", "BMI_Recalculated"]].head()

In [ ]:
# insert() places a column at a specific position.
health.insert(3, "Age_Group", np.where(health["Age"] < 18, "Teen", "Adult"))
health.head()

In [ ]:
# drop() removes columns or rows.
health = health.drop(columns=["BMI_Recalculated"])
health.head()

## 12. `map`, `apply`, `where`, and `assign`

Prefer vectorized operations when possible. Use these methods when transformations need more structure.

In [ ]:
# map() replaces values in a Series using a dictionary.
students["Gender"].map({"Male": "M", "Female": "F"})

In [ ]:
# apply() applies a function to each value.
def pass_label(mark):
    if mark >= 70:
        return "Pass"
    return "Review"

students["Marks"].apply(pass_label)

In [ ]:
# where() keeps original values where the condition is True.
students["Marks"].where(students["Marks"] >= 70, other=0)

In [ ]:
# assign() is useful in method chains because it returns a new DataFrame.
students.assign(
    Marks_Percent=lambda df: df["Marks"] / 100,
    Result=lambda df: np.where(df["Marks"] >= 70, "Pass", "Review")
)

## 13. Missing Values

Missing values appear as `NaN`, `None`, or `NaT` depending on the dtype.

In [ ]:
survey = pd.DataFrame({
    "Name": ["Aarav", "Aditi", "Vihaan", "Diya", "Kabir"],
    "Age": [17, np.nan, 17, 19, np.nan],
    "Marks": [86, 78, np.nan, 91, 67],
    "City": ["Bengaluru", None, "Delhi", "Bengaluru", None]
})

survey

In [ ]:
survey.isna().sum()

In [ ]:
# Drop rows with any missing value.
survey.dropna()

In [ ]:
filled_survey = survey.copy()

# Fill numeric values with mean or median.
filled_survey["Age"] = filled_survey["Age"].fillna(filled_survey["Age"].mean())
filled_survey["Marks"] = filled_survey["Marks"].fillna(filled_survey["Marks"].median())

# Fill text values with a clear label.
filled_survey["City"] = filled_survey["City"].fillna("Unknown")

filled_survey

In [ ]:
# Forward fill uses the previous valid value.
survey.ffill()

## 14. Duplicates and Cleaning

Real data often has duplicate rows, extra spaces, inconsistent case, and invalid numeric values.

In [ ]:
dirty_students = pd.DataFrame({
    "Name": [" Aarav ", "Aditi", "Aditi", "vihaan", "Diya"],
    "City": ["bengaluru", "Mumbai", "Mumbai", "delhi", "Bengaluru"],
    "Marks": [86, 78, 78, 72, 91]
})

dirty_students

In [ ]:
# Check duplicate rows.
dirty_students.duplicated()

In [ ]:
clean_students = dirty_students.drop_duplicates().copy()

# Clean string columns.
clean_students["Name"] = clean_students["Name"].str.strip().str.title()
clean_students["City"] = clean_students["City"].str.strip().str.title()

# Keep marks inside a valid range.
clean_students["Marks"] = clean_students["Marks"].clip(lower=0, upper=100)

clean_students

## 15. Data Types

Correct data types make calculations safer and memory usage better.

In [ ]:
type_demo = pd.DataFrame({
    "Student_ID": ["1", "2", "3"],
    "Marks": ["86", "78", "missing"],
    "Joined_On": ["2026-01-10", "2026-01-11", "not available"],
    "Level": ["Normal", "Overweight", "Normal"]
})

type_demo

In [ ]:
type_demo["Student_ID"] = type_demo["Student_ID"].astype("int64")
type_demo["Marks"] = pd.to_numeric(type_demo["Marks"], errors="coerce")
type_demo["Joined_On"] = pd.to_datetime(type_demo["Joined_On"], errors="coerce")
type_demo["Level"] = type_demo["Level"].astype("category")

type_demo.dtypes

## 16. String Operations

Use `.str` to apply string methods to a whole column.

In [ ]:
names = pd.DataFrame({
    "Full_Name": ["Aarav Sharma", "Aditi Rao", "Vihaan Mehta", "Diya Iyer"],
    "Email": ["aarav@example.com", "aditi@example.com", "vihaan@test.org", "diya@test.org"]
})

names

In [ ]:
# Split full name into first and last name.
names[["First_Name", "Last_Name"]] = names["Full_Name"].str.split(" ", expand=True)

# Filter text and extract email domain.
names["Domain"] = names["Email"].str.extract(r"@(.+)$")
names[names["Email"].str.contains("example", case=False, na=False)]

In [ ]:
names

## 17. Date and Time

Use `pd.to_datetime()` to convert text dates. Then use `.dt` to access date parts.

In [ ]:
attendance = pd.DataFrame({
    "Student_ID": [1, 1, 2, 2, 3, 3],
    "Date": ["2026-01-01", "2026-01-08", "2026-01-01", "2026-01-08", "2026-01-01", "2026-01-08"],
    "Hours_Studied": [2.5, 3.0, 1.5, 2.0, 4.0, 3.5]
})

attendance["Date"] = pd.to_datetime(attendance["Date"])
attendance

In [ ]:
attendance["Year"] = attendance["Date"].dt.year
attendance["Month"] = attendance["Date"].dt.month_name()
attendance["Day_Name"] = attendance["Date"].dt.day_name()

attendance

In [ ]:
# Resample weekly after setting a datetime index.
attendance_by_date = attendance.set_index("Date").sort_index()
attendance_by_date.resample("W")["Hours_Studied"].sum()

## 18. GroupBy and Aggregation

`groupby()` splits data into groups, applies calculations, and combines results.

In [ ]:
# Average marks by city.
students.groupby("City")["Marks"].mean()

In [ ]:
# Multiple aggregations.
students.groupby("City")["Marks"].agg(["count", "mean", "min", "max"])

In [ ]:
# Named aggregations create clean column names.
obesity_data.groupby("Obesity_Level").agg(
    students_count=("Student_ID", "count"),
    avg_bmi=("BMI", "mean"),
    avg_weight=("Weight_kg", "mean"),
    avg_sleep=("Sleep_hours", "mean"),
    max_calories=("Daily_calorie_intake", "max")
)

In [ ]:
# Group by multiple columns.
obesity_data.groupby(["Gender", "Obesity_Level"]).agg(
    count=("Student_ID", "count"),
    avg_bmi=("BMI", "mean"),
    avg_marks=("Marks", "mean")
)

## 19. GroupBy `transform` and `filter`

`transform()` returns one value per original row. `filter()` keeps or removes whole groups.

In [ ]:
student_compare = obesity_data.copy()

student_compare["Avg_BMI_By_Level"] = student_compare.groupby("Obesity_Level")["BMI"].transform("mean")
student_compare["BMI_Diff_From_Level_Avg"] = student_compare["BMI"] - student_compare["Avg_BMI_By_Level"]

student_compare[["Name", "Obesity_Level", "BMI", "Avg_BMI_By_Level", "BMI_Diff_From_Level_Avg"]].head()

In [ ]:
# Keep only obesity-level groups whose average BMI is above 25.
high_bmi_groups = obesity_data.groupby("Obesity_Level").filter(lambda group: group["BMI"].mean() > 25)
high_bmi_groups["Obesity_Level"].value_counts()

## 20. Reshaping With `melt`

`melt()` converts wide data into long data.

In [ ]:
wide_scores = pd.DataFrame({
    "Student_ID": [1, 2, 3],
    "Math": [88, 75, 70],
    "Science": [82, 80, 74],
    "English": [90, 84, 79]
})

wide_scores

In [ ]:
long_scores = wide_scores.melt(
    id_vars="Student_ID",
    var_name="Subject",
    value_name="Score"
)

long_scores

## 21. Pivot, Pivot Table, and Crosstab

Use these for summaries and wide-format tables.

In [ ]:
# pivot() works when each Student_ID and Subject pair has one value.
long_scores.pivot(index="Student_ID", columns="Subject", values="Score")

In [ ]:
exam_attempts = pd.DataFrame({
    "Student_ID": [1, 1, 1, 2, 2, 2, 2],
    "Subject": ["Math", "Math", "Science", "Math", "Science", "Science", "English"],
    "Score": [88, 92, 82, 75, 80, 85, 84]
})

# pivot_table() handles duplicate combinations by aggregating.
exam_attempts.pivot_table(
    index="Student_ID",
    columns="Subject",
    values="Score",
    aggfunc="mean",
    fill_value=0
)

In [ ]:
# Crosstab counts category combinations.
pd.crosstab(obesity_data["Gender"], obesity_data["Obesity_Level"])

In [ ]:
# Row percentages.
pd.crosstab(
    obesity_data["Gender"],
    obesity_data["Obesity_Level"],
    normalize="index"
).round(2)

## 22. Concat

Use `concat()` to stack DataFrames vertically or horizontally.

In [ ]:
class_a = pd.DataFrame({
    "Student_ID": [1, 2],
    "Name": ["Aarav", "Aditi"],
    "Class": ["A", "A"]
})

class_b = pd.DataFrame({
    "Student_ID": [3, 4],
    "Name": ["Vihaan", "Diya"],
    "Class": ["B", "B"]
})

pd.concat([class_a, class_b], ignore_index=True)

In [ ]:
# Horizontal concat joins side-by-side by index.
contact_info = pd.DataFrame({
    "Email": ["aarav@example.com", "aditi@example.com", "vihaan@example.com", "diya@example.com"]
})

pd.concat([pd.concat([class_a, class_b], ignore_index=True), contact_info], axis=1)

## 23. Joins and Merges

`merge()` combines rows using matching key columns.

Common join types:

- Inner join: keep matching keys from both tables.
- Left join: keep all rows from the left table.
- Right join: keep all rows from the right table.
- Outer join: keep all rows from both tables.
- Cross join: create every possible row pair.
- Semi join: keep left rows that have a match in right.
- Anti join: keep left rows that do not have a match in right.
- Self join: join a table with itself.

In [ ]:
student_master = pd.DataFrame({
    "Student_ID": [1, 2, 3, 4, 5],
    "Name": ["Aarav", "Aditi", "Vihaan", "Diya", "Kabir"],
    "City": ["Bengaluru", "Mumbai", "Delhi", "Bengaluru", "Pune"]
})

exam_results = pd.DataFrame({
    "Student_ID": [1, 2, 3, 6],
    "Math": [88, 75, 70, 95],
    "Science": [82, 80, 74, 89]
})

display(student_master)
display(exam_results)

### Inner Join

Keeps only keys that exist in both DataFrames.

In [ ]:
pd.merge(student_master, exam_results, on="Student_ID", how="inner")

### Left Join

Keeps all rows from the left DataFrame.

In [ ]:
pd.merge(student_master, exam_results, on="Student_ID", how="left")

### Right Join

Keeps all rows from the right DataFrame.

In [ ]:
pd.merge(student_master, exam_results, on="Student_ID", how="right")

### Outer Join

Keeps all keys from both DataFrames. Missing matches become `NaN`.

In [ ]:
pd.merge(student_master, exam_results, on="Student_ID", how="outer", indicator=True)

### Cross Join

Creates every possible combination of rows.

In [ ]:
products = pd.DataFrame({"Product": ["Notebook", "Pen"]})
colors = pd.DataFrame({"Color": ["Blue", "Black", "Red"]})

pd.merge(products, colors, how="cross")

### Semi Join and Anti Join

Pandas does not have direct `semi` or `anti` keywords, but they are simple to create.

In [ ]:
# Semi join: keep left rows that have a match.
student_master[student_master["Student_ID"].isin(exam_results["Student_ID"])]

In [ ]:
# Anti join: keep left rows that do not have a match.
student_master[~student_master["Student_ID"].isin(exam_results["Student_ID"])]

In [ ]:
# Anti join using merge indicator. Useful for multiple join keys.
merged = pd.merge(student_master, exam_results, on="Student_ID", how="left", indicator=True)
merged[merged["_merge"] == "left_only"].drop(columns=["Math", "Science", "_merge"])

### Different Column Names

Use `left_on` and `right_on` when key names differ.

In [ ]:
fees = pd.DataFrame({
    "ID": [1, 2, 4],
    "Fees_Paid": [True, False, True]
})

pd.merge(student_master, fees, left_on="Student_ID", right_on="ID", how="left")

### Index Join

Use `.join()` when keys are indexes.

In [ ]:
students_indexed = student_master.set_index("Student_ID")
results_indexed = exam_results.set_index("Student_ID")

students_indexed.join(results_indexed, how="left")

### Self Join

A self join finds relationships inside one table.

In [ ]:
employees = pd.DataFrame({
    "Employee_ID": [1, 2, 3, 4],
    "Employee": ["Neha", "Rohan", "Isha", "Kabir"],
    "Manager_ID": [np.nan, 1, 1, 2]
})

employees.merge(
    employees[["Employee_ID", "Employee"]],
    left_on="Manager_ID",
    right_on="Employee_ID",
    how="left",
    suffixes=("", "_Manager")
).rename(columns={"Employee_Manager": "Manager"})[["Employee_ID", "Employee", "Manager_ID", "Manager"]]

### Merge Validation and Suffixes

Use `validate` to catch unexpected duplicate matches.

In [ ]:
student_contacts = pd.DataFrame({
    "Student_ID": [1, 2, 3, 4, 5],
    "Name": ["A. Sharma", "A. Rao", "V. Mehta", "D. Iyer", "K. Singh"],
    "Phone": ["111", "222", "333", "444", "555"]
})

pd.merge(
    student_master,
    student_contacts,
    on="Student_ID",
    how="left",
    suffixes=("_master", "_contact"),
    validate="one_to_one"
)

## 24. `combine_first` and `update`

Use these to fill or replace values from another DataFrame.

In [ ]:
primary = pd.DataFrame({
    "Student_ID": [1, 2, 3],
    "Email": ["aarav@example.com", np.nan, "vihaan@example.com"]
}).set_index("Student_ID")

backup = pd.DataFrame({
    "Student_ID": [1, 2, 3],
    "Email": ["aarav.backup@example.com", "aditi@example.com", "vihaan.backup@example.com"]
}).set_index("Student_ID")

primary.combine_first(backup)

In [ ]:
primary_copy = primary.copy()
primary_copy.update(backup)
primary_copy

## 25. Indexes and MultiIndex

Indexes label rows. MultiIndex stores multiple index levels.

In [ ]:
students_by_id = students.set_index("Student_ID")
students_by_id.loc[3]

In [ ]:
multi_summary = obesity_data.groupby(["Gender", "Obesity_Level"])[["BMI", "Marks"]].mean()
multi_summary

In [ ]:
# Select from a MultiIndex.
multi_summary.loc[("Male", "Overweight")]

In [ ]:
# unstack moves an index level into columns. stack reverses it.
multi_summary.unstack("Gender")

## 26. Window Functions

Window functions calculate values across nearby rows.

In [ ]:
weekly_health = pd.DataFrame({
    "Week": pd.date_range("2026-01-01", periods=8, freq="W"),
    "Steps": [42000, 46000, 39000, 50000, 52000, 48000, 56000, 59000],
    "Calories": [15500, 16000, 15000, 17000, 17200, 16800, 18000, 18500]
})

weekly_health

In [ ]:
weekly_health["Steps_3W_Rolling_Mean"] = weekly_health["Steps"].rolling(window=3).mean()
weekly_health["Calories_Expanding_Mean"] = weekly_health["Calories"].expanding().mean()
weekly_health["Cumulative_Steps"] = weekly_health["Steps"].cumsum()

weekly_health

## 27. Ranking and Binning

Ranking orders rows. Binning converts numbers into categories.

In [ ]:
ranked = obesity_data[["Name", "BMI", "Marks"]].copy()

ranked["Marks_Rank"] = ranked["Marks"].rank(ascending=False, method="dense")
ranked.sort_values("Marks_Rank").head()

In [ ]:
bmi_bins = [0, 18.5, 25, 30, 100]
bmi_labels = ["Underweight", "Normal", "Overweight", "Obese"]

ranked["BMI_Category_From_Cut"] = pd.cut(ranked["BMI"], bins=bmi_bins, labels=bmi_labels)
ranked["Marks_Quartile"] = pd.qcut(ranked["Marks"], q=4, labels=["Q1", "Q2", "Q3", "Q4"])

ranked.head()

## 28. Basic Plotting

Pandas plotting is useful for quick exploration.

In [ ]:
obesity_data["Obesity_Level"].value_counts().plot(kind="bar", title="Obesity Level Counts")

In [ ]:
obesity_data.plot(kind="scatter", x="Height_cm", y="Weight_kg", title="Height vs Weight")

In [ ]:
obesity_data["BMI"].plot(kind="hist", bins=10, title="BMI Distribution")

## 29. Method Chaining

Method chaining keeps transformations readable from top to bottom.

In [ ]:
clean_summary = (
    obesity_data
    .assign(
        BMI_Recalculated=lambda df: df["Weight_kg"] / (df["Height_cm"] / 100) ** 2,
        Age_Group=lambda df: np.where(df["Age"] < 18, "Teen", "Adult")
    )
    .query("Daily_calorie_intake >= 2200")
    .groupby(["Age_Group", "Obesity_Level"], as_index=False)
    .agg(
        count=("Student_ID", "count"),
        avg_bmi=("BMI_Recalculated", "mean"),
        avg_exercise=("Exercise_hours_per_week", "mean")
    )
    .sort_values(["Age_Group", "avg_bmi"], ascending=[True, False])
)

clean_summary

## 30. Performance and Best Practices

- Prefer vectorized operations over loops.
- Select only the columns you need.
- Use `category` for repeated text values with few unique values.
- Avoid chained assignment. Use `.loc[row_condition, column] = value`.
- Use `copy()` when you intentionally create an independent DataFrame.
- Use `merge(validate=...)` when you know the expected relationship.
- Keep raw data unchanged and create cleaned copies.

In [ ]:
# Good assignment pattern with .loc.
best_practice_demo = obesity_data.copy()
best_practice_demo.loc[best_practice_demo["Age"] >= 18, "Age_Group"] = "Adult"
best_practice_demo.loc[best_practice_demo["Age"] < 18, "Age_Group"] = "Teen"

best_practice_demo[["Name", "Age", "Age_Group"]].head()

In [ ]:
# Convert repeated text columns to category to save memory.
optimized = obesity_data.copy()
for column in ["Gender", "Family_history_obesity", "Obesity_Level"]:
    optimized[column] = optimized[column].astype("category")

optimized.info()

## 31. Mini End-to-End Analysis

This combines cleaning, feature creation, grouping, sorting, and review.

In [ ]:
analysis = (
    obesity_data
    .copy()
    .assign(
        BMI_Recalculated=lambda df: (df["Weight_kg"] / (df["Height_cm"] / 100) ** 2).round(2),
        Lifestyle_Score=lambda df: (
            df["Exercise_hours_per_week"] * 2
            + df["Sleep_hours"]
            + df["Water_liters_per_day"]
            - df["Fast_food_meals_per_week"]
        ).round(2)
    )
)

analysis[["Name", "BMI", "BMI_Recalculated", "Lifestyle_Score", "Obesity_Level"]].head()

In [ ]:
level_analysis = (
    analysis
    .groupby("Obesity_Level", as_index=False)
    .agg(
        students=("Student_ID", "count"),
        avg_bmi=("BMI_Recalculated", "mean"),
        avg_lifestyle_score=("Lifestyle_Score", "mean"),
        avg_marks=("Marks", "mean")
    )
    .sort_values("avg_bmi", ascending=False)
)

level_analysis

## 32. Practice Tasks

1. Select only `Name`, `Age`, `BMI`, and `Obesity_Level` from `obesity_data`.
2. Filter students with `BMI > 25` and `Exercise_hours_per_week < 3`.
3. Create a column called `Hydration_Level` using `Water_liters_per_day`.
4. Find average `Marks` by `Gender` and `Obesity_Level`.
5. Create a pivot table with `Obesity_Level` as rows, `Gender` as columns, and average `BMI` as values.
6. Create a small DataFrame with `Student_ID` and `Club`, then left join it with `obesity_data`.
7. Use an anti join to find students from `obesity_data` who are not in your club DataFrame.
8. Convert `Gender` and `Obesity_Level` to category dtype.
9. Sort students by `BMI` descending and show the top 10.
10. Build one method chain that creates a feature, filters rows, groups data, and sorts the result.

In [ ]:
# Practice area: write your solutions below.